# 🌐 Notebook 08 — Streamlit App: Full Beginner Guide + Portfolio Tips
**Project:** Sales & Retail Analytics Portfolio
**Author:** Muhammad Umar Shahid
**App location:** `streamlit_app/`
**Run command:** `streamlit run app.py` from `streamlit_app/` directory

---

## 📋 What This Notebook Covers

| Section | Topic |
|---|---|
| 1 | What is Streamlit and why use it |
| 2 | Project structure explained |
| 3 | How Streamlit works (execution model) |
| 4 | Library explanations |
| 5 | File-by-file code walkthrough |
| 6 | How models are loaded and used |
| 7 | UI components used in this app |
| 8 | How to run, deploy, and share |
| 9 | Portfolio presentation tips |

---

## 🎯 Section 1 — What is Streamlit?

**Streamlit** is a Python library that turns plain Python scripts into
interactive web applications — with zero HTML, CSS, or JavaScript required.

### Why Streamlit for a Data Science Portfolio?

| Tool | What it needs | Who builds it |
|---|---|---|
| Flask / Django | Python + HTML + CSS + JS | Full-stack developer |
| Dash | Python + React knowledge | Data engineer |
| Power BI | Drag and drop, no code | Business analyst |
| **Streamlit** | **Python only** | **Data scientist ✅** |

Streamlit is the industry standard for data scientists who want to
showcase ML models and dashboards without becoming web developers.

### How Streamlit Executes

This is the most important concept to understand:

> **Every time a user interacts with a widget (button, slider, dropdown),
> Streamlit reruns the entire Python script from top to bottom.**

This is different from Flask/Django where code runs once.
It means:
- You write top-to-bottom code like a regular Python script
- Streamlit handles all the reactivity for you
- `@st.cache_data` and `@st.cache_resource` prevent expensive operations
  from rerunning unnecessarily on every interaction

## 🗂️ Section 2 — Project Structure

```
streamlit_app/
│
├── app.py                    ← Landing page and entry point
│
├── pages/                    ← Each file = one page in the sidebar
│   ├── 01_overview.py        ← Page 1: Sales KPIs and charts
│   ├── 02_profitability.py   ← Page 2: Profit analysis
│   ├── 03_ml_forecast.py     ← Page 3: Prophet forecasting
│   └── 04_ml_predict.py      ← Page 4: XGBoost + LightGBM
│
├── utils/                    ← Shared helper modules
│   ├── data_loader.py        ← Load CSV + load ML models
│   └── styles.py             ← CSS + reusable UI components
│
└── .streamlit/
    └── config.toml           ← Theme colors and settings
```

### Why This Structure?

**Multi-page apps** in Streamlit work by placing files in a `pages/`
folder. Streamlit automatically:
- Detects them and adds them to the sidebar navigation
- Names them based on the filename (underscores become spaces)
- Orders them by the number prefix (01, 02, 03...)

**`utils/`** follows the **DRY principle** (Don't Repeat Yourself):
- `data_loader.py` — load data and models once, share everywhere
- `styles.py` — define CSS once, apply to all pages

Without `utils/`, you would copy-paste the same 50 lines into every page.

## 📦 Section 3 — Library Explanations

| Library | Version | Purpose in This App |
|---|---|---|
| `streamlit` | 1.56.0 | Core framework — creates all UI components |
| `plotly` | 6.6.0 | Interactive charts (hover, zoom, filter) |
| `prophet` | 1.3.0 | Facebook's time-series forecasting model |
| `xgboost` | 3.2.0 | Gradient boosted trees for profit regression |
| `lightgbm` | 4.6.0 | Fast gradient boosting for churn classification |
| `scikit-learn` | 1.8.0 | train_test_split, metrics (MAE, RMSE, R²) |
| `pandas` | — | All data manipulation |
| `numpy` | — | Numerical operations, array handling |
| `pickle` | built-in | Load saved model files from disk |
| `os` | built-in | Build file paths that work on any OS |

### Why Plotly over Matplotlib?

| Feature | Matplotlib | Plotly |
|---|---|---|
| Interactivity | ❌ Static | ✅ Hover, zoom, pan |
| Streamlit integration | Needs `st.pyplot()` | `st.plotly_chart()` |
| Appearance | Basic | Professional |
| Export | PNG only | PNG + HTML + SVG |

In a web app context, Plotly is always preferred over Matplotlib
because users can interact with charts directly in the browser.

## 🔧 Section 4 — `utils/data_loader.py` Explained

This is the most important file in the app.
Every single page imports from it.

### The `@st.cache_data` Decorator

```python
@st.cache_data
def load_data():
    df = pd.read_csv(path)
    return df
```

**Without caching:**
- User clicks a filter → script reruns → CSV reloads from disk → slow

**With `@st.cache_data`:**
- First run → loads CSV → stores result in memory
- Every subsequent run → returns stored result instantly
- Cache resets only when the function code changes

**When to use `@st.cache_data` vs `@st.cache_resource`:**

| Decorator | Use for | Example |
|---|---|---|
| `@st.cache_data` | Data (DataFrames, lists, dicts) | `load_data()` |
| `@st.cache_resource` | Objects (models, DB connections) | `load_xgb_model()` |

`@st.cache_resource` is used for ML models because they are large
objects that should be shared across all users and sessions,
not copied like data.

### Path Building with `os.path`

```python
def get_project_root():
    return os.path.dirname(      # 3. go up to project root
               os.path.dirname(  # 2. go up to streamlit_app/
                   os.path.dirname(  # 1. go up to utils/
                       os.path.abspath(__file__)  # start: data_loader.py
                   )
               )
           )
```

This builds the path dynamically so the app works on any machine,
regardless of where the project is saved. Never hardcode paths like
`C:/Users/umars/...` in production code — it breaks on other machines.

## 🎨 Section 5 — `utils/styles.py` Explained

### Why CSS in a Python File?

Streamlit's default theme is limited. We use `st.markdown()` with
`unsafe_allow_html=True` to inject custom CSS directly into the page.

```python
st.markdown("""
<style>
    [data-testid="stMetric"] {
        background-color: #f0f4f8;
        border-left: 4px solid #2e6da4;
        border-radius: 8px;
    }
</style>
""", unsafe_allow_html=True)
```

`[data-testid="stMetric"]` is a CSS selector that targets Streamlit's
internal metric component. These selectors are found by inspecting
the browser's developer tools (F12 → Elements tab).

### The `page_header()` Function

```python
def page_header(icon, title, subtitle):
    st.markdown(f"""
    <div style='background: linear-gradient(135deg, #1a3a5c, #2e6da4);
                padding: 1.8rem; border-radius: 12px;'>
        <h1 style='color:#ffffff;'>{icon} {title}</h1>
        <p style='color:#b0c4de;'>{subtitle}</p>
    </div>
    """, unsafe_allow_html=True)
```

This is a **reusable component** — called once per page with different
arguments. Instead of copying the same HTML into 4 files, you call:

```python
page_header("📊", "Sales Overview", "KPI metrics · Sales trends")
```

This is the same principle as functions in any programming language —
write once, use everywhere.

## 📊 Section 6 — `pages/01_overview.py` Explained

### Page Setup Pattern

Every page follows this exact pattern:

```python
# 1. Page config — must be FIRST Streamlit command
st.set_page_config(page_title="Overview", page_icon="📊", layout="wide")

# 2. Apply CSS
apply_styles()

# 3. Load data
df = load_data()

# 4. Render sidebar filters
segment, region, category = sidebar_filters(df)

# 5. Filter data based on selections
filtered = filter_data(df, segment, region, category)

# 6. Render page content
page_header(...)
st.metric(...)
st.plotly_chart(...)
```

`st.set_page_config()` must always be the very first Streamlit call —
before any other `st.*` command. This is a Streamlit requirement.

### KPI Metrics

```python
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Orders", f"{len(filtered):,}")
```

`st.columns(4)` creates 4 equal-width columns side by side.
`st.metric()` renders a styled KPI card with a label and value.
The `:,` format specifier adds comma separators to numbers (8286 → 8,286).

### Area Chart with Plotly

```python
fig1 = px.area(monthly, x='Month', y='Sales')
fig1.update_traces(fill='tozeroy', fillcolor='rgba(46,109,164,0.15)')
fig1.update_layout(plot_bgcolor="white", paper_bgcolor="white")
st.plotly_chart(fig1, use_container_width=True)
```

- `px.area()` — creates an area chart (line chart with shaded fill below)
- `fill='tozeroy'` — fills the area down to zero on the y-axis
- `rgba(46,109,164,0.15)` — same blue as the line but 15% opacity
- `plot_bgcolor="white"` — white chart background
- `use_container_width=True` — chart expands to fill available width

## 💰 Section 7 — `pages/02_profitability.py` Explained

### Scatter Plot with Trendline

```python
fig1 = px.scatter(
    scatter_data,
    x         = 'Discount',
    y         = 'Profit Margin %',
    color     = 'Category',
    opacity   = 0.5,
    trendline = "ols"
)
```

- `opacity=0.5` — 50% transparent points to show overlapping data
- `trendline="ols"` — adds Ordinary Least Squares regression line
  per category automatically
- `color='Category'` — different color per category

`trendline="ols"` requires `pip install statsmodels` — it's already
installed in your venv from Notebook 06.

### Box Plot

```python
fig2 = px.box(filtered, x='Region', y='Profit', color='Region')
fig2.add_hline(y=0, line_dash="dash", line_color="red")
```

A box plot shows 5 statistics in one chart:
- Bottom whisker = minimum (excluding outliers)
- Bottom of box = 25th percentile (Q1)
- Middle line = median (50th percentile)
- Top of box = 75th percentile (Q3)
- Top whisker = maximum (excluding outliers)
- Dots = outliers

`add_hline(y=0)` adds a horizontal reference line at zero — the
break-even point. Orders below this line are loss-making.

## 📈 Section 8 — `pages/03_ml_forecast.py` Explained

### Loading a Pre-trained Model

```python
prophet_model = load_prophet_model()
```

This loads the model that was already trained in Notebook 03.
No training happens here — just inference (prediction).

**Training vs Inference:**

| Step | Where | Time |
|---|---|---|
| Training | Notebook 03 | Minutes to hours |
| Saving | Notebook 03 | `pickle.dump()` |
| Loading | Streamlit app | < 1 second |
| Predicting | Streamlit app | Milliseconds |

This separation is **production best practice**:
- Train offline when you have time and computing resources
- Serve predictions online instantly to users

### Generating a Forecast

```python
future   = prophet_model.make_future_dataframe(periods=periods, freq='MS')
forecast = prophet_model.predict(future)
```

- `make_future_dataframe()` — creates a blank dataframe of future dates
- `freq='MS'` — monthly frequency, start of month
- `predict()` — fills in `yhat` (forecast), `yhat_lower`, `yhat_upper`

No retraining — we just extend the timeline and ask the model
to predict values for the new dates.

### Confidence Interval Shading

```python
fig.add_trace(go.Scatter(
    x         = pd.concat([forecast_only['ds'], forecast_only['ds'][::-1]]),
    y         = pd.concat([forecast_only['yhat_upper'],
                           forecast_only['yhat_lower'][::-1]]),
    fill      = "toself",
    fillcolor = "rgba(225,87,89,0.1)",
))
```

This is a common technique for drawing a shaded confidence band:
1. Plot the upper bound left to right
2. Plot the lower bound right to left (reversed with `[::-1]`)
3. Connect the ends to form a closed polygon
4. Fill the polygon with a semi-transparent color

The result is a shaded region between `yhat_lower` and `yhat_upper`.

## 🤖 Section 9 — `pages/04_ml_predict.py` Explained

### Tab 1: XGBoost Profit Prediction

#### Applying Saved Encoders

```python
for col, encoder in encoders.items():
    if col in model_df.columns:
        known         = set(encoder.classes_)
        model_df[col] = model_df[col].apply(
            lambda x: x if x in known else encoder.classes_[0]
        )
        model_df[col] = encoder.transform(model_df[col].astype(str))
```

The encoders were saved in Notebook 03. We must apply the **exact same
encoding** as training — if the model learned that `West=3`, we must
pass `3` not `West`.

The `lambda` handles **unseen labels** gracefully:
if a new value appears that wasn't in the training data,
we replace it with the first known class instead of crashing.

#### Why Use `train_test_split` if Model is Already Trained?

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size/100, random_state=42
)
y_pred = xgb_model.predict(X_test)
```

We split the data only to get a held-out test set for **evaluation**.
We never call `.fit()` — only `.predict()`.
This lets users see how the model performs on unseen data
by adjusting the test size slider.

### Tab 2: LightGBM Churn Prediction

#### Why Rebuild RFM Features?

```python
customer_features = df.groupby('Customer ID').agg(
    Total_Orders = ('Order ID', 'nunique'),
    ...
)
```

We can't load `churn_risk_scores.csv` here because the churn threshold
is a dynamic slider — the user can change it from 90 to 365 days.
We rebuild RFM features live so the churn predictions update instantly
when the threshold changes.

This is a good example of **interactive ML** — the model stays fixed
but the labels and evaluations change with user input.

#### Churn Risk Tiers

```python
def assign_risk_tier(prob):
    if prob >= 0.75:   return '🔴 Critical Risk'
    elif prob >= 0.50: return '🟠 High Risk'
    elif prob >= 0.25: return '🟡 Medium Risk'
    else:              return '🟢 Low Risk'
```

Raw probabilities (0.734, 0.821...) are hard for business users to act on.
Translating them into **named tiers** is standard practice in production
churn systems — marketing teams can immediately know which customers
need urgent outreach vs routine retention.

## ⚙️ Section 10 — `.streamlit/config.toml` Explained

```toml
[theme]
primaryColor             = "#2e6da4"
backgroundColor          = "#ffffff"
secondaryBackgroundColor = "#f0f4f8"
textColor                = "#1a1a2e"
font                     = "sans serif"
```

`config.toml` sets the global theme for the entire app:

| Key | Effect |
|---|---|
| `primaryColor` | Accent color for buttons, sliders, active elements |
| `backgroundColor` | Main page background |
| `secondaryBackgroundColor` | Sidebar and card backgrounds |
| `textColor` | All text color |
| `font` | Font family across the app |

These values override Streamlit's default purple theme with your
corporate blue palette — making the app look professional and
consistent with your Power BI dashboard colors.

## 🚀 Section 11 — How to Run the App

### Locally

```bash
# Step 1 — Activate venv
cd C:/Users/umars/Downloads/sales-retail-analytics
venv\Scripts\activate

# Step 2 — Navigate to app folder
cd streamlit_app

# Step 3 — Run
streamlit run app.py
```

App opens at `http://localhost:8501` automatically.

### Stopping the App

Press `Ctrl + C` in the terminal.

### Sharing with Others

**Option 1 — Streamlit Community Cloud (Free)**
1. Push project to GitHub
2. Go to share.streamlit.io
3. Connect your GitHub repo
4. Set main file as `streamlit_app/app.py`
5. Get a public URL to share with recruiters

**Option 2 — Share the GitHub Repo**
Recruiters can clone and run locally — include a `README.md` with
clear run instructions.

**Option 3 — Screen recording**
Record a 2-3 minute walkthrough of the app and include it in your
portfolio or LinkedIn.

## 🎯 Section 12 — Portfolio Presentation Tips

### What Recruiters Look for in a Streamlit App

| What they see | What it signals |
|---|---|
| Clean, consistent UI | Attention to detail, professional output |
| Pre-trained models loading fast | Understands training vs inference |
| Filters that work correctly | Data manipulation skills |
| Proper error handling | Production mindset |
| Clear labelling and metrics | Communication skills |
| Multi-page structure | Software engineering thinking |

### How to Present This App in Interviews

**"Tell me about your portfolio project"** — mention the app:
> "I built an end-to-end analytics platform in Streamlit that serves
> four pre-trained ML models — Prophet for forecasting, XGBoost for
> profit regression, LightGBM for churn classification, and KMeans
> for customer segmentation. The app loads models from disk rather
> than retraining, which reflects how production ML systems work."

**"What would you improve?"** — show you think beyond the task:
> "I'd add authentication so different users see different data,
> implement a database connection instead of a CSV file,
> and add monitoring to track prediction drift over time."

### What to Include in Your GitHub README

```markdown
## 🌐 Streamlit App

Multi-page interactive dashboard covering:
- 📊 Sales Overview — KPIs, trends, category analysis
- 💰 Profitability — Discount impact, profit margins
- 📈 ML Forecast — Prophet 12-month sales forecasting
- 🤖 ML Predictions — XGBoost profit + LightGBM churn

**Run locally:**
pip install -r requirements.txt
cd streamlit_app
streamlit run app.py
```

### Common Interview Questions About Streamlit Apps

**Q: Why Streamlit over Flask?**
> Streamlit is purpose-built for data science — no HTML/CSS required,
> built-in caching, and native integration with pandas and plotly.
> Flask is better for REST APIs and traditional web apps.

**Q: How does caching work in your app?**
> I used `@st.cache_data` for the dataset and `@st.cache_resource`
> for ML models. The distinction is important — cache_data copies
> data per session while cache_resource shares model objects across
> all users, which is more memory-efficient for large models.

**Q: How would you deploy this to production?**
> For a real deployment I'd use Streamlit Community Cloud for quick
> sharing, or containerise with Docker and deploy to AWS EC2 or
> Azure App Service for enterprise use. I'd also add environment
> variables for any sensitive config instead of hardcoded paths.

**Q: What's the biggest limitation of your app?**
> It reads from a static CSV file. In production, this would connect
> to a live database like PostgreSQL, so the dashboard always shows
> current data without manual file updates.

## ✅ Section 13 — Final Summary

### Files Created

| File | Purpose |
|---|---|
| `app.py` | Landing page and navigation |
| `pages/01_overview.py` | Sales KPIs and trends |
| `pages/02_profitability.py` | Profit and discount analysis |
| `pages/03_ml_forecast.py` | Prophet forecasting |
| `pages/04_ml_predict.py` | XGBoost + LightGBM predictions |
| `utils/data_loader.py` | Shared data and model loading |
| `utils/styles.py` | Shared CSS and UI components |
| `.streamlit/config.toml` | App theme configuration |

### Models Used (Pre-trained from Notebook 03)

| Model | File | Task |
|---|---|---|
| Prophet | `prophet_model.pkl` | Sales forecasting |
| XGBoost | `xgboost_profit_model.pkl` | Profit regression |
| LightGBM | `lightgbm_churn_model.pkl` | Churn classification |

### Key Concepts Demonstrated

- **Caching** — `@st.cache_data` and `@st.cache_resource`
- **Multi-page apps** — `pages/` folder convention
- **Reusable components** — `utils/styles.py`
- **Training vs inference** — pre-trained models loaded from disk
- **Dynamic ML** — churn threshold slider changes predictions live
- **Professional UI** — custom CSS, consistent color palette
- **Portfolio quality** — clean code, comments, logical structure

```python
# ── Git Commands ──────────────────────────────────────────────
print("""
Git update commands:

    cd sales-retail-analytics
    git add .
    git commit -m "Add Notebook 08: Streamlit app explanation and portfolio guide"
    git push origin main

Next steps:
    → Notebook 08 complete
    → Kaggle publish
    → CV update
""")
```